## Step-1: Setup and Authentication

In [2]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import datetime

In [3]:
ee.Authenticate()
ee.Initialize(project = "striking-berm-241917")

## Step-2: Define the AOI

In [19]:
import geopandas as gpd
gdf = gpd.read_file("./Boundary/Pulaski_Boundary.shp") #This shapefile is in the local drive and is the area of interest for my research. 

In [20]:
#This code shows the Coordinate Reference System of the shapefile. 
print("CRS:", gdf.crs)
print("Rows:", len(gdf))
gdf.head()

CRS: EPSG:4326
Rows: 1


,POPULATION,POP_SQMI,SQMI,POPULATI_1,POP20_SQMI,Shape__Are,Shape__Len,NAME,STATE_NAME,STATE_ABBR,STATE_FIPS,COUNTY_FIP,FIPS,geometry
0,402304,498.31,807.34,399125,494.4,0.2059,2.840276,Pulaski County,Arkansas,AR,05,119,05119,"POLYGON ((-92.12155 35.01221, -92.12004 35.012..."


In [10]:
#Here we convert the shapefile into json file format which gee package can read and display in geemap. 
#The shapefile is not directly supported so we need to covert into json
import json
geojson = json.loads(gdf.to_json())
pulaski_fc = ee.FeatureCollection(geojson)
aoi = pulaski_fc.geometry()

print("Area:", round(aoi.area().divide(1e6).getInfo(), 1))

Area: 2091.2


In [12]:
m = geemap.Map()
m.centerObject(aoi, 9)
m.addLayer(aoi, {"color":"red"}, "Pulaski County Boundary")
m


Map(center=[34.769972921563394, -92.31184050630443], controls=(WidgetControl(options=['position', 'transparent…

## Step-3: Building the Landsat 8/9 Image Collection (2013-2026)

In [15]:
import datetime

start_date = "2013-01-01"
end_date = "2026-10-19"

l8 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
)

l9 = (
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    .filterBounds(aoi)
    .filterDate(start_date, end_date)
)

landsat = l8.merge(l9).sort("system:time_start")
print(f"Scenes over Pulaski County, {start_date} to {end_date}:", landsat.size().getInfo())

Scenes over Pulaski County, 2013-01-01 to 2026-10-19: 694


## Step -4: Cloud-mask and scale each image

In [17]:
def mask_scale(image):
    qa = image.select("QA_PIXEL")
    cloud_bit = 1<<3
    shadow_bit = 1<<4
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
        qa.bitwiseAnd(shadow_bit).eq(0)
    )

    optical_bands = image.select("SR_B.").multiply(0.0000275).add(-0.0000275)

    thermal_band = image.select("ST_B10").multiply(0.00341802).add(149.0)

    return (
        image.addBands(optical_bands, None, True)
        .addBands(thermal_band, None, True)
        .updateMask(mask)
        .copyProperties(image, ["system: time_start"])
    )

landsat_clean = landsat.map(mask_scale)
print("Scenes after mapping mask_scale: ", landsat_clean.size().getInfo())

Scenes after mapping mask_scale:  694


## Step-5: Clip Every image to the Pulaski County Boundary

In [18]:
def clip_to_aoi(image):
    return image.clip(aoi).copyProperties(image, ["system:time_start"])

landsat_clipped = landsat_clean.map(clip_to_aoi)

print("Scenes after clipping: ", landsat_clipped.size().getInfo())


Scenes after clipping:  694


## Step-6: Visualization

In [29]:
# Pick a reasonably clear summer scene to look atuple


landsat_path24 = (
    landsat_clipped
    .filter(ee.Filter.eq("WRS_PATH", 24))
    .filter(ee.Filter.eq("WRS_ROW", 36))
)

print("Scenes in path 24 / row 36:", landsat_path24.size().getInfo())

sample_image = (
    landsat_path24
    .filter(ee.Filter.calendarRange(5, 8, "month"))
    .filter(ee.Filter.lt("CLOUD_COVER", 10))
    .sort("CLOUD_COVER")
    .first()
)

print("Date of sample image:", sample_image.date().format("YYYY-MM-dd").getInfo())
print("Cloud cover (%):", sample_image.get("CLOUD_COVER").getInfo())


#View based on the true %colors
true_color_vis = {
    "bands":["SR_B4","SR_B3", "SR_B2"],   #Red, Green, Blue
    "min":0,
    "max":0.3,
}


#Thermal band visualization
thermal_vis = {
    "bands":["ST_B10"],   #Red, Green, Blue
    "min":295,
    "max":315,
    "palette":["blue","yellow","orange","red"],
}

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(sample_image, true_color_vis, "True Color (Clipped)")
m.addLayer(sample_image, thermal_vis, "Thermal Band Visualization (Clipped")
m.addLayer(aoi, {"color":"black"}, "Pulaski County Boundary", opacity=0.5)
m

Scenes in path 24 / row 36: 350
Date of sample image: 2024-06-13
Cloud cover (%): 0.01


Map(center=[34.76997292156348, -92.31184050630442], controls=(WidgetControl(options=['position', 'transparent_…

## Step-7: Build one summer composite per year